# 14. Computer Vision Data Pipeline

Learn the seven basics of preparing image data, using the local CIFAR-10 dataset. No data is downloaded.

**Pipeline:** Raw images → Validation → Preprocessing → Augmentation → Dataset → DataLoader → Model


## 1. Folder-based datasets

`ImageFolder` reads class names from folders. Put images for each label in its own folder, for example `train/cat/` and `train/dog/`. This makes a folder dataset easy to reuse without writing a custom loader.


## 2. Class mapping

Each class folder gets a number for model training. `ImageFolder.class_to_idx` shows the mapping, such as `{'airplane': 0, 'automobile': 1, ...}`. Keep the same class folders in every split so labels stay consistent.


## 3. Train, validation, and test folders

Use the same class subfolders under `train/`, `val/`, and `test/`. Train data fits the model, validation data helps select settings, and test data is saved for the final evaluation. Never use test images for training.

This short setup creates a small, balanced folder example from the project's local CIFAR-10 files.


In [1]:
from pathlib import Path
from PIL import Image
from torchvision.datasets import CIFAR10
import numpy as np
data = Path("data")
if not (data / "cifar-10-batches-py").exists(): raise FileNotFoundError("Local CIFAR-10 data is missing.")
train_raw, test_raw = CIFAR10(data, train=True, download=False), CIFAR10(data, train=False, download=False)
root = data / "cifar10_pipeline_demo"
for label, name in enumerate(train_raw.classes):
    train_ids = np.where(np.array(train_raw.targets) == label)[0][:30]
    test_ids = np.where(np.array(test_raw.targets) == label)[0][:10]
    for split, source, ids in [("train", train_raw, train_ids[:20]), ("val", train_raw, train_ids[20:]), ("test", test_raw, test_ids)]:
        folder = root / split / name
        folder.mkdir(parents=True, exist_ok=True)
        for n, i in enumerate(ids): Image.fromarray(source.data[i]).save(folder / f"{n}.png")
print("Created train, val, and test folders under", root)

Created train, val, and test folders under data\cifar10_pipeline_demo


## 4. Image validation

Check that files open and have the expected size and color mode before training. Invalid dimensions or modes may need conversion to match the model's input.


## 5. Corrupted image detection

A corrupted file cannot be decoded as an image. The check below verifies each file and flags corrupt files or unexpected image formats before loading the dataset.


In [2]:
bad, sizes, modes = [], set(), set()
for path in root.rglob("*.png"):
    try:
        with Image.open(path) as image:
            sizes.add(image.size); modes.add(image.mode)
            if image.size != (32, 32) or image.mode != "RGB": bad.append(str(path))
            image.verify()
    except (OSError, ValueError): bad.append(str(path))
print(f"Images: {len(list(root.rglob('*.png')))} | sizes: {sizes} | modes: {modes} | invalid/corrupt: {len(set(bad))}")
if bad: raise ValueError(f"Check these image files: {bad[:5]}")

Images: 800 | sizes: {(32, 32)} | modes: {'RGB'} | invalid/corrupt: 0


## 6. Class imbalance

A class imbalance means some labels have many more images than others. Compare per-class counts; if counts differ greatly, consider collecting more examples or using class-weighted training. This demo is intentionally balanced.


## 7. Dataset statistics

Useful basic statistics include the number of images in each split, images per class, and image dimensions. These checks can reveal missing classes, uneven splits, or unexpected image sizes.


In [3]:
from torchvision.datasets import ImageFolder
import pandas as pd
train = ImageFolder(root / "train")
val, test = ImageFolder(root / "val"), ImageFolder(root / "test")
print("Class mapping:", train.class_to_idx)
counts = pd.DataFrame({s: [len(list((root / s / c).glob("*.png"))) for c in train.classes]
    for s in ["train", "val", "test"]}, index=train.classes)
display(counts.rename_axis("Class"))
print(f"Split totals: train={len(train)}, val={len(val)}, test={len(test)} | image size={sizes}")

Class mapping: {'airplane': 0, 'automobile': 1, 'bird': 2, 'cat': 3, 'deer': 4, 'dog': 5, 'frog': 6, 'horse': 7, 'ship': 8, 'truck': 9}


,train,val,test
Class,,,
airplane,40,20,20
automobile,40,20,20
bird,40,20,20
cat,40,20,20
deer,40,20,20
dog,40,20,20
frog,40,20,20
horse,40,20,20
ship,40,20,20


Split totals: train=400, val=200, test=200 | image size={(32, 32)}


## Finish the pipeline: preprocessing, augmentation, DataLoader, and model

Resize and normalize every image. Apply random flipping only to training data. The DataLoader groups images into batches, which the simple model accepts.


In [4]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import transforms
norm = transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
train.transform = transforms.Compose([transforms.Resize((96, 96)), transforms.RandomHorizontalFlip(), transforms.ToTensor(), norm])
val.transform = test.transform = transforms.Compose([transforms.Resize((96, 96)), transforms.ToTensor(), norm])
loader = DataLoader(train, batch_size=32, shuffle=True)
images, labels = next(iter(loader))
model = nn.Sequential(nn.Conv2d(3, 8, 3, padding=1), nn.ReLU(), nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(8, len(train.classes)))
print("Batch:", images.shape, "| model output:", model(images).shape)

Batch: torch.Size([32, 3, 96, 96]) | model output: torch.Size([32, 10])
